# ASR Assignment 2025-26

This notebook has been provided as a template to get you started on the assignment.  Feel free to use it for your development, or do your development directly in Python.

You can find a full description of the assignment [here](https://www.inf.ed.ac.uk/teaching/courses/asr/coursework-2026.html).

You are provided with two Python modules `observation_model.py` and `wer.py`.  The first was described in [Lab 3](https://github.com/geoph9/asr_labs/blob/main/asr_lab3_4.ipynb).  The second can be used to compute the number of substitution, deletion and insertion errors between ASR output and a reference text.

It can be used as follows:

```python
import wer

my_refence = 'A B C'
my_output = 'A C C D'

wer.compute_alignment_errors(my_reference, my_output)
```

This produces a tuple $(s,d,i)$ giving counts of substitution,
deletion and insertion errors respectively - in this example (1, 0, 1).  The function accepts either two strings, as in the example above, or two lists.  Matching is case sensitive.

## Template code

Assuming that you have already made a function to generate an WFST, `create_wfst()` and a decoder class, `MyViterbiDecoder`, you can perform recognition on all the audio files as follows:


In [4]:
import glob
import os
import wer
import observation_model
import openfst_python as fst
from decoder import MyViterbiDecoder
from utils import generate_word_wfst, parse_lexicon, generate_symbol_tables, generate_phone_wfst
import math
import time
from collections import Counter

def compute_unigram_probs(wav_files):  # ADDED: count word frequencies from transcriptions
    counts = Counter()
    for wav_file in wav_files:
        transcription_file = os.path.splitext(wav_file)[0] + '.txt'
        with open(transcription_file, 'r') as f:
            words = f.readline().strip().split()
        counts.update(words)
    total = sum(counts.values())
    return {word: count / total for word, count in counts.items()}

def create_wfst(lexicon, word_table, phone_table, state_table, unigram_probs, n_states=3, stay_cost=0.9, final_prob=0.1):  # ADDED unigram_probs
    f = fst.Fst('log')
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)

    loop_state = f.add_state()
    f.set_start(loop_state)
    f.set_final(loop_state)

    trans_cost = 1.0 - stay_cost

    for word, phones in lexicon.items():
        word_id = word_table.find(word)
        current_state = loop_state

        # ADDED: get unigram prob for this word, fall back to uniform if unseen
        prob = unigram_probs.get(word, 1.0 / len(lexicon))
        unigram_weight = -math.log(prob)

        for phone_idx, phone in enumerate(phones):
            for i in range(1, n_states + 1):
                state_sym = f"{phone}_{i}"
                in_label = state_table.find(state_sym)

                sl_weight = fst.Weight('log', -math.log(stay_cost))
                f.add_arc(current_state, fst.Arc(in_label, 0, sl_weight, current_state))

                next_state = f.add_state()

                # ADDED: apply unigram weight on first arc of each word only
                if phone_idx == 0 and i == 1:
                    fw_weight = fst.Weight('log', -math.log(trans_cost) + unigram_weight)
                else:
                    fw_weight = fst.Weight('log', -math.log(trans_cost))

                if phone_idx == len(phones) - 1 and i == n_states:
                    out_label = word_id
                else:
                    out_label = 0

                f.add_arc(current_state, fst.Arc(in_label, out_label, fw_weight, next_state))
                current_state = next_state

        final_weight = fst.Weight('log', -math.log(final_prob))
        f.add_arc(current_state, fst.Arc(0, 0, final_weight, loop_state))

    return f


def read_transcription(wav_file):
    transcription_file = os.path.splitext(wav_file)[0] + '.txt'
    with open(transcription_file, 'r') as f:
        transcription = f.readline().strip()
    return transcription


lex = parse_lexicon("lexicon.txt")
word_table, phone_table, state_table = generate_symbol_tables(lex)

# COMMENTED OUT: stay_cost sweep (best stay=0.9 found previously)
# for stay_cost in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:

# COMMENTED OUT: final_prob sweep (best final=0.1 found previously)
# for final_prob in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:

stay_cost = 0.9   # Fixed at best value from stay sweep
final_prob = 0.1  # Fixed at best value from final_prob sweep

wav_files = glob.glob('/group/teaching/asr/labs/recordings/*.wav')

# ADDED: compute unigram probabilities from transcriptions
unigram_probs = compute_unigram_probs(wav_files)
print(f"Computed unigram probs for {len(unigram_probs)} unique words\n")
print(f"{'Word':<20} {'Probability':>12} {'Log-prob':>12}")
print("-" * 46)
for word, prob in sorted(unigram_probs.items(), key=lambda x: -x[1]):
    print(f"  {word:<18} {prob:>12.6f} {-math.log(prob):>12.6f}")
print()


f = create_wfst(lex, word_table, phone_table, state_table, unigram_probs, stay_cost=stay_cost, final_prob=final_prob)

num_states = f.num_states()
num_arcs = sum(1 for s in f.states() for _ in f.arcs(s))
print(f"=== WFST Size (stay={stay_cost:.1f}, final={final_prob:.1f}, unigram LM) ===")
print(f"  States : {num_states}")
print(f"  Arcs   : {num_arcs}\n")

total_substitutions = 0
total_deletions = 0
total_insertions = 0
total_words = 0
total_forward_computations = 0
total_decode_time = 0.0
total_backtrace_time = 0.0

for wav_file in wav_files:

    decoder = MyViterbiDecoder(f, wav_file)

    t0 = time.perf_counter()
    decoder.decode()
    decode_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    (state_path, words) = decoder.backtrace()
    backtrace_time = time.perf_counter() - t0

    total_decode_time += decode_time
    total_backtrace_time += backtrace_time
    total_forward_computations += decoder.forward_computation_count

    words_str = ' '.join(words)
    transcription = read_transcription(wav_file)
    error_counts = wer.compute_alignment_errors(transcription, words_str)
    word_count = len(transcription.split())

    total_substitutions += error_counts[0]
    total_deletions += error_counts[1]
    total_insertions += error_counts[2]
    total_words += word_count

overall_wer = (total_substitutions + total_deletions + total_insertions) / total_words

print(f"=== Overall Results (stay={stay_cost:.1f}, final={final_prob:.1f}, unigram LM) ===")
print(f"  WER                        : {overall_wer:.2%}")
print(f"  Total substitutions        : {total_substitutions}")
print(f"  Total deletions            : {total_deletions}")
print(f"  Total insertions           : {total_insertions}")
print(f"  Total decode time          : {total_decode_time:.4f}s")
print(f"  Total backtrace time       : {total_backtrace_time:.4f}s")
print(f"  Total time                 : {total_decode_time + total_backtrace_time:.4f}s")

Computed unigram probs for 10 unique words

Word                  Probability     Log-prob
----------------------------------------------
  peppers                0.124925     2.080042
  peck                   0.119520     2.124276
  peter                  0.118318     2.134377
  picked                 0.118318     2.134377
  piper                  0.114715     2.165307
  pickled                0.109910     2.208094
  of                     0.107508     2.230195
  the                    0.067868     2.690193
  a                      0.064264     2.744752
  where's                0.054655     2.906721

=== WFST Size (stay=0.9, final=0.1, unigram LM) ===
  States : 106
  Arcs   : 220



Exception: 